# 1. Extract and review

Published Aily clients are initialized only by the CLI task. Set the tenant/environment, authenticated cloud profile, region and workbook runtime in your shell before starting. Reads are read-only and bounded. All candidates require approval in Excel. No amounts or likes are editable inputs.


In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd

ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists()
)
OUT = Path(os.environ.get("FMI_OUTPUT_DIR", str(ROOT / "outputs/fmi-controlled")))
OUT.mkdir(parents=True, exist_ok=True)
INPUTS = OUT / "FMI_inputs.xlsx"
SNAPSHOT = OUT / "snapshot.json"


def command(*args):
    subprocess.run(
        [sys.executable, "-m", "aily_project.cli", "fmi", *map(str, args)],
        cwd=ROOT,
        check=True,
    )

In [ ]:
if not INPUTS.exists():
    command(
        "seed",
        "--prospects",
        ROOT
        / "scripts/fashion-momentum-index/handover/brand_peers_dinner prospects.xlsx",
        "--legacy",
        ROOT / "scripts/fashion-momentum-index/inputs",
        "--output",
        INPUTS,
    )
command(
    "extract",
    "--inputs",
    INPUTS,
    "--output",
    SNAPSHOT,
    "--first-year",
    os.environ.get("FMI_FIRST_YEAR", "2022"),
    "--last-year",
    os.environ.get("FMI_LAST_YEAR", "2026"),
)
snapshot = json.loads(SNAPSHOT.read_text())
display(pd.DataFrame(snapshot["metadata"].get("candidate_mappings", [])))
display(pd.DataFrame(snapshot["metadata"].get("queries", [])))
print("\n".join(snapshot["metadata"]["limitations"]))